## **Default Chat Model Setup**

In [1]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

google_model="gemini-3.1-flash-lite"

In [2]:
# LLM Schema
from typing import Literal
from pydantic import BaseModel, Field

class llm_schema(BaseModel):
    movie_review_flag: Literal["positive", "negative"] = Field(..., description="The sentiment of the movie review, either 'positive' or 'negative'.")
    movie_review_post: str = Field(..., description="The post content of the movie review.")



## **Fundamental Process**

In [3]:
# Task 1 : Prompt Generation with Dynamic Inputs
from langchain_core.prompts import ChatPromptTemplate

dynamic_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a movie review sentiment analyzer. You will receive a movie review and your task is to determine whether the review is positive or negative. Additionally, you will generate a social media post based on the review content."),
    ("user", "Bhool Bhulaiyaa 3 is the worst movie I have ever seen. The plot is nonsensical, the acting is wooden, and the special effects are laughable. I can't believe I wasted two hours of my life on this disaster. Avoid it at all costs.")
])

In [4]:
# Task 2 : LLM
from langchain.chat_models import init_chat_model

llm_gemini = init_chat_model(model=google_model,model_provider="openai",
    openai_api_key=os.environ["GOOGLE_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

llm_structured_output = llm_gemini.with_structured_output(llm_schema)

In [5]:
# Task 3 : Schema Output Parser
from langchain_core.output_parsers import StrOutputParser

str_output_parser = StrOutputParser()
llm_structured_output.invoke("Bhool Bhulaiyaa 3 is the worst movie I have ever seen. The plot is nonsensical, the acting is wooden, and the special effects are laughable. I can't believe I wasted two hours of my life on this disaster. Avoid it at all costs.")

llm_schema(movie_review_flag='negative', movie_review_post="Bhool Bhulaiyaa 3 is the worst movie I have ever seen. The plot is nonsensical, the acting is wooden, and the special effects are laughable. I can't believe I wasted two hours of my life on this disaster. Avoid it at all costs.")

In [6]:
# Task 4 : Custom Runnable
from langchain_core.runnables import RunnableLambda

def custom_runnable(schema:llm_schema) -> dict:
    return {"movie_review_flag": schema.movie_review_flag, "content": schema.movie_review_post}

custom_runnable_instance = RunnableLambda(custom_runnable)

## **Conditional Chain 1**

In [7]:

instagram_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a instagram content creator. You are very chill and use a lot of slang. You are also very funny and sarcastic."),
    ("user", "Give me a readymade Instagram post: {content}. Make sure to include hashtags and emojis. Only give me the post, do not include any other text.")
])

instagram_chain = instagram_prompt | llm_gemini | str_output_parser


## **Conditional Chain 2**

In [8]:
def generate_linkedin_post(content_dict: dict) -> str:
    content = content_dict.get("content", "")
    linkedin_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a LinkedIn content creator. You are very professional and use a lot of industry-specific terminology."),
        ("user", "Give me a readymade LinkedIn post: {content}. Make sure to include a professional tone and relevant hashtags. Only give me the post, do not include any other text.")
    ])

    linkedin_chain = linkedin_prompt | llm_gemini | str_output_parser

    response = linkedin_chain.invoke({"content": content})

    return response

## **Custom Conditional Chain**

In [9]:
# Task 5 : Conditional Chain
from langchain_core.runnables import RunnableBranch, RunnableLambda
conditional_chain =  dynamic_prompt | llm_structured_output | custom_runnable_instance | RunnableBranch( (lambda x : x["movie_review_flag"] == "positive", RunnableLambda(generate_linkedin_post)),  instagram_chain )

In [10]:
conditional_chain.invoke({})

'Just watched Bhool Bhulaiyaa 3 and… yeah, I’d rather watch paint dry, no cap. 💀 The plot? Literally non-existent. The acting? Giving “I’m only here for the paycheck.” Save your coins and your sanity, besties, this one is a major skip. 🚩📉 Stay home, scroll TikTok, do literally anything else. ✌️🙄\n\n#BhoolBhulaiyaa3 #MovieReview #SaveYourMoney #HardPass #Cringe #Bollywood #WasteOfTime #CinemaFail'